In [1]:
import gc
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.inspection import permutation_importance

In [2]:
df = pd.read_csv("E:\\UofT\\05_Research\\00_File\\output\\data\\ml_oilgas_panel_1km_2018_2024.csv")


In [3]:
# parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# remove impossible rows
df = df.dropna(subset=["grid_id", "date", "flare_label"]).copy()

# target must be integer
df["flare_label"] = df["flare_label"].astype(int)

In [4]:

model_feature_cols = [
    # spatial
    "lon", "lat",

    # raw bands
    "B4_med", "B8_med", "B11_med",

    # raw satellite variables
    "ndvi", "ndbi", "ntl", "ntl_log", "lst_night_c", "s2_count", "cf_cvg",

    # engineered features
    "builtup_index",
    "nir_red_ratio", "swir_nir_ratio", "swir_red_ratio",
    "heat_light_interaction", "built_light_interaction", "heat_built_interaction",
    "oil_signature_score",

    # anomalies
    "ntl_anom", "lst_anom", "ndvi_anom", "ndbi_anom", "b11_anom", "swir_nir_anom",

    # temporal features
    "ntl_roll3", "ntl_roll6",
    "lst_roll3", "lst_roll6",
    "b11_roll3", "swir_nir_roll3",
    "ntl_change", "lst_change", "b11_change", "ndvi_change",
    "ntl_persist6", "lst_persist6", "b11_persist6", "swir_nir_persist6",

    # seasonality
    "month_sin", "month_cos"
]

model_feature_cols = [c for c in model_feature_cols if c in df.columns]



In [5]:
# memory optimization: keep only necessary columns
target_col = "flare_label"

base_keep_cols = [
    "grid_id", "year", "month", "date", "lon", "lat",
    target_col, "oilgas_candidate"
]

keep_cols = list(dict.fromkeys(base_keep_cols + model_feature_cols))
keep_cols = [c for c in keep_cols if c in df.columns]

df = df[keep_cols]

In [6]:
# Downcast integers
int_cols = df.select_dtypes(include=["int64", "int32"]).columns
for col in int_cols:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# Downcast floats
float_cols = df.select_dtypes(include=["float64"]).columns
for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")

# Compact target / date-related fields
if "flare_label" in df.columns:
    df["flare_label"] = df["flare_label"].astype("int8")

if "year" in df.columns:
    df["year"] = df["year"].astype("int16")

if "month" in df.columns:
    df["month"] = df["month"].astype("int8")

print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Memory usage (MB): 824.79


In [7]:
# ------------------------------------------------------------
# Add missing indicators
# ------------------------------------------------------------
missing_indicator_cols = []

for col in model_feature_cols:
    miss_col = f"{col}_isna"
    df[miss_col] = df[col].isna().astype("int8")
    missing_indicator_cols.append(miss_col)

all_feature_cols = model_feature_cols + missing_indicator_cols

print("Total features after missing indicators:", len(all_feature_cols))
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Total features after missing indicators: 84
Memory usage (MB): 994.6


In [8]:
train_mask = df["year"] <= 2022
test_mask  = df["year"] >= 2023

print("Train rows:", int(train_mask.sum()))
print("Test rows:", int(test_mask.sum()))

X_train = df.loc[train_mask, all_feature_cols]
X_test  = df.loc[test_mask, all_feature_cols]

y_train = df.loc[train_mask, "flare_label"]
y_test  = df.loc[test_mask, "flare_label"]


Train rows: 3028200
Test rows: 1211280


In [9]:
# Free temporary mask memory
del train_mask, test_mask
gc.collect()

0

In [10]:
# model and fit

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = n_neg / n_pos

sample_weight = np.where(y_train == 1, pos_weight, 1.0)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        max_depth=6,
        min_samples_leaf=40,
        l2_regularization=1.0,
        random_state=42
    ))
])

model.fit(X_train, y_train, clf__sample_weight=sample_weight)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('clf',
                 HistGradientBoostingClassifier(l2_regularization=1.0,
                                                learning_rate=0.05, max_depth=6,
                                                max_iter=250,
                                                min_samples_leaf=40,
                                                random_state=42))])

In [11]:
# Evaluate model
pred_test = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, pred_test)
pr_auc = average_precision_score(y_test, pred_test)

print("ROC-AUC:", roc_auc)
print("PR-AUC :", pr_auc)

ROC-AUC: 0.9994450765488684
PR-AUC : 0.8409188353014474


In [12]:
# THRESHOLD
precision, recall, thresholds = precision_recall_curve(y_test, pred_test)

f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold by F1:", best_threshold)
print("Best F1:", f1_scores[best_idx])

Best threshold by F1: 0.9772445141557637
Best F1: 0.7683749157108959


In [13]:
def evaluate_threshold(y_true, pred_prob, threshold):
    pred_label = (pred_prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision": precision_score(y_true, pred_label, zero_division=0),
        "recall": recall_score(y_true, pred_label, zero_division=0),
        "f1": f1_score(y_true, pred_label, zero_division=0),
        "predicted_positives": int(pred_label.sum())
    }

threshold_list = [0.50, 0.70, 0.80, 0.90, float(best_threshold)]
threshold_results = pd.DataFrame(
    [evaluate_threshold(y_test, pred_test, t) for t in threshold_list]
).sort_values("threshold")

threshold_results

,threshold,precision,recall,f1,predicted_positives
0,0.500000,0.205029,0.994194,0.339951,14198
1,0.700000,0.306912,0.991803,0.468765,9462
2,0.800000,0.382583,0.984290,0.550999,7533
3,0.900000,0.508285,0.963798,0.665566,5552
4,0.977245,0.758655,0.778347,0.768375,3004


In [14]:
pred_label_best = (pred_test >= best_threshold).astype(int)

print("Classification report:")
print(classification_report(y_test, pred_label_best, digits=3))

print("Confusion matrix:")
print(confusion_matrix(y_test, pred_label_best))

Classification report:
              precision    recall  f1-score   support

           0      0.999     0.999     0.999   1208352
           1      0.759     0.778     0.768      2928

    accuracy                          0.999   1211280
   macro avg      0.879     0.889     0.884   1211280
weighted avg      0.999     0.999     0.999   1211280

Confusion matrix:
[[1207627     725]
 [    649    2279]]


In [15]:
# Permutation importance

# Use a subsample for speed
sample_n = min(100000, len(X_test))

sample_idx = np.random.RandomState(42).choice(len(X_test), size=sample_n, replace=False)

X_test_sample = X_test.iloc[sample_idx]
y_test_sample = y_test.iloc[sample_idx]

perm = permutation_importance(
    model,
    X_test_sample,
    y_test_sample,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
    n_jobs=-1
)

perm_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("Top 20 permutation importance features:")
perm_importance.head(20)

Top 20 permutation importance features:


,feature,importance_mean,importance_std
0,lon,0.665669,0.020960
1,lat,0.558197,0.013372
7,ntl,0.336418,0.010742
20,ntl_anom,0.081211,0.004201
27,ntl_roll6,0.060287,0.003259
15,swir_red_ratio,0.050266,0.003611
26,ntl_roll3,0.019225,0.002633
8,ntl_log,0.017086,0.002891
6,ndbi,0.013843,0.002921
70,lst_roll3_isna,0.013393,0.002655


In [16]:
# Score full panel
chunk_size = 300000
pred_parts = []

for start in range(0, len(df), chunk_size):
    end = min(start + chunk_size, len(df))
    X_chunk = df.iloc[start:end][all_feature_cols]
    pred_chunk = model.predict_proba(X_chunk)[:, 1]
    pred_parts.append(pred_chunk)
    print(f"Scored rows {start:,} to {end:,}")

df["pred_prob"] = np.concatenate(pred_parts)

print(df[["grid_id", "year", "month", "pred_prob"]].head())

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,300,000
Scored rows 3,300,000 to 3,600,000
Scored rows 3,600,000 to 3,900,000
Scored rows 3,900,000 to 4,200,000
Scored rows 4,200,000 to 4,239,480
   grid_id  year  month  pred_prob
0        1  2018      1   0.000684
1        1  2018      2   0.000571
2        1  2018      3   0.000887
3        1  2018      4   0.000523
4        1  2018      5   0.000596


In [17]:
# monthly activity flags
strict_threshold = max(0.98, float(best_threshold))
discovery_threshold = 0.70

df["active_flag_strict"] = (df["pred_prob"] >= strict_threshold).astype("int8")
df["active_flag_discovery"] = (df["pred_prob"] >= discovery_threshold).astype("int8")

print("Strict threshold   :", strict_threshold)
print("Discovery threshold:", discovery_threshold)

Strict threshold   : 0.98
Discovery threshold: 0.7


In [18]:
# Aggregate to grid level
# -------------------------------------------------------------------
grid_scores = (
    df.groupby("grid_id", as_index=False)
      .agg(
          lon=("lon", "mean"),
          lat=("lat", "mean"),
          flare_label_ever=("flare_label", "max"),
          mean_risk=("pred_prob", "mean"),
          max_risk=("pred_prob", "max"),
          strict_active_months=("active_flag_strict", "sum"),
          discovery_active_months=("active_flag_discovery", "sum")
      )
)

grid_scores.head()

,grid_id,lon,lat,flare_label_ever,mean_risk,max_risk,strict_active_months,discovery_active_months
0,1,45.212429,34.507957,0,0.000733,0.005152,0,0
1,2,45.223324,34.507938,0,0.001030,0.006211,0,0
2,3,45.234219,34.507915,0,0.000358,0.000358,0,0
3,4,45.212452,34.516975,0,0.002509,0.007872,0,0
4,5,45.223351,34.516956,0,0.002772,0.007857,0,0


In [19]:

strict_candidates = (
    grid_scores[
        (grid_scores["flare_label_ever"] == 0) &
        (grid_scores["max_risk"] >= strict_threshold) &
        (grid_scores["strict_active_months"] >= 2)
    ]
    .sort_values(["max_risk", "strict_active_months"], ascending=[False, False])
)

discovery_candidates = (
    grid_scores[
        (grid_scores["flare_label_ever"] == 0) &
        (grid_scores["max_risk"] >= discovery_threshold) &
        (grid_scores["discovery_active_months"] >= 3)
    ]
    .sort_values(["max_risk", "discovery_active_months"], ascending=[False, False])
)

print("Strict candidates:", len(strict_candidates))
print("Discovery candidates:", len(discovery_candidates))

strict_candidates.head(20)

Strict candidates: 139
Discovery candidates: 926


,grid_id,lon,lat,flare_label_ever,mean_risk,max_risk,strict_active_months,discovery_active_months
14294,14295,44.212364,35.425358,0,0.990025,0.998899,81,84
9502,9503,44.422874,35.255226,0,0.960316,0.998688,35,60
16748,16749,44.332787,35.516247,0,0.982450,0.998619,54,84
16749,16750,44.343815,35.516308,0,0.988233,0.998606,70,84
32482,32483,43.782742,36.152054,0,0.986903,0.998336,47,60
20902,20903,45.005527,35.671375,0,0.946455,0.998270,14,60
17231,17232,44.354698,35.534401,0,0.984589,0.998197,62,84
63064,63065,45.005524,35.671375,0,0.970234,0.998137,12,24
15775,15776,44.145702,35.479000,0,0.918757,0.998111,17,84
16990,16991,44.354771,35.525383,0,0.987140,0.997953,67,84


In [20]:
if "oilgas_candidate" in df.columns:
    heuristic_grid = (
        df.groupby("grid_id", as_index=False)
          .agg(oilgas_candidate_ever=("oilgas_candidate", "max"))
    )

    strict_candidates_filtered = strict_candidates.merge(
        heuristic_grid,
        on="grid_id",
        how="left"
    )

    strict_candidates_filtered = strict_candidates_filtered[
        strict_candidates_filtered["oilgas_candidate_ever"] == 1
    ].copy()

    print("Strict candidates after heuristic filter:", len(strict_candidates_filtered))
    strict_candidates_filtered.head(20)
else:
    strict_candidates_filtered = pd.DataFrame()
    print("oilgas_candidate not found, skipping heuristic filter.")

Strict candidates after heuristic filter: 86


In [22]:
import os

# ------------------------------------------------------------
# Define base output directory (custom path)
# ------------------------------------------------------------
base_output_dir = r"E:\UofT\05_Research\00_File\output\data"

# Subfolders
pred_dir = os.path.join(base_output_dir, "predictions")
cand_dir = os.path.join(base_output_dir, "candidates")
diag_dir = os.path.join(base_output_dir, "diagnostics")

# Create directories
os.makedirs(pred_dir, exist_ok=True)
os.makedirs(cand_dir, exist_ok=True)
os.makedirs(diag_dir, exist_ok=True)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
df.to_csv(os.path.join(pred_dir, "grid_month_predictions.csv"), index=False)
grid_scores.to_csv(os.path.join(pred_dir, "grid_level_scores.csv"), index=False)

strict_candidates.to_csv(os.path.join(cand_dir, "strict_candidates.csv"), index=False)
discovery_candidates.to_csv(os.path.join(cand_dir, "discovery_candidates.csv"), index=False)

if not strict_candidates_filtered.empty:
    strict_candidates_filtered.to_csv(
        os.path.join(cand_dir, "strict_candidates_filtered.csv"),
        index=False
    )

perm_importance.to_csv(os.path.join(diag_dir, "feature_importance.csv"), index=False)
threshold_results.to_csv(os.path.join(diag_dir, "threshold_comparison.csv"), index=False)

print("Saved all outputs to:", base_output_dir)

KeyboardInterrupt: 